
# 17 — Your own images: phone photos (Real) and ChatGPT images (AI) → v3

The final system (v1+v2) was never trained on **modern phone photos** or **ChatGPT (OpenAI GPT Image)** images, so it gets both wrong.
This notebook adds them:

| Folder | Label |
|---|---|
| `own_data/phone_real/` | **Real** — untouched phone photos |
| `own_data/chatgpt/` | **Full AI** — downloaded ChatGPT images, generated or edited (ChatGPT redraws the whole picture, so every pixel is generated) |

1. Images are made upright and resized exactly as the app does; near-duplicates are grouped so they never cross splits.
2. Split by group: about 55% train, 20% validation, **25% test (never used for training or for any choice)**.
3. The v2 unified model is fine-tuned (8 epochs) on all old sources plus your images → **v3 unified**.
4. The decision is refitted on validation (v1 networks + v3 unified + v2 edit finder), chosen exactly as in notebook 16.
5. v1+v2 and v3 are compared on the main test set, notebook 14's sets and **your held-out test images**.

Saves `final_hybrid_v3_deploy.pkl` → `python app.py --v3`. Nothing earlier is overwritten. About 45–60 minutes.
To retrain after adding more images, just run it again (it notices the new images).


In [1]:
# ============================================================
# 1. Setup + configuration
# ============================================================
import os, sys, io, json, math, time, random, pickle, subprocess, importlib, warnings
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
def ensure(module, pip_name=None):
    try: __import__(module)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])
for m in ["segmentation_models_pytorch:segmentation-models-pytorch", "datasets", "diffusers", "accelerate", "transformers"]:
    mod, _, pipn = m.partition(":"); ensure(mod, pipn or None)
import numpy as np, pandas as pd, torch, torch.nn.functional as F, matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from PIL import Image
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
warnings.filterwarnings("ignore"); Image.MAX_IMAGE_PIXELS = None
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

PROJECT_DIR = (Path("/content/drive/MyDrive/AI_Image_Project") if IN_COLAB
               else Path(os.environ.get("AI_PROJECT_DIR", r"C:\AI_Image_Project")))   # <- laptop folder
BASE_DIR = PROJECT_DIR / "Partial_AI_5000"
FEATURES_DIR, LOC_FEATURES_DIR = PROJECT_DIR / "features", PROJECT_DIR / "features" / "localization"
LOC_MODELS_DIR = PROJECT_DIR / "saved_models" / "localization"
EXTRA = PROJECT_DIR / "extra_train"; OUT_DIR = PROJECT_DIR / "results" / "v2"
EXTRA.mkdir(exist_ok=True); OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = dict(
    SEED=42, MAX_SIDE=1024,
    ARCH="Unet", ENCODER="resnet34",
    V1_UNIFIED="unified_Unet_resnet34_crop384", V1_LOC="Unet_resnet34_crop512_plusbbox_clean",   # v1 models (start points)
    COCOAI_SKIP_ROWS=200, COCOAI_ROWS=600, TRAIN_GENERATORS=["sd2.1", "sdxl", "sd3"],          # SD3.5, DALL-E 3, Midjourney stay unseen
    N_SD_TRAIN=600, N_SD_VAL=60,
    SD_PROMPTS=["a photo", "an object", "a person", "an animal", "a plate of food", "a car", "a plant", "a bag", "a chair", "a sign"],
    # localization fine-tune (Flux + SD edits)
    LOC_CROP=512, LOC_EPOCHS=15, LOC_SAMPLES=2000, LOC_LR=1e-4, LOC_SHARE={"partial": 0.5, "sdinpaint": 0.5},
    # unified fine-tune (all sources)
    UNI_CROP=384, UNI_EPOCHS=15, UNI_SAMPLES=3000, UNI_LR=1e-4,
    UNI_SHARE={"coco": 1/9, "original": 1/9, "cocoai_real": 1/9, "diffusiondb": 1/6, "cocoai_fake": 1/6, "partial": 1/6, "sdinpaint": 1/6},
    SCALE_PROB=0.5, SCALE_RANGE=(0.6, 1.0), JPEG_PROB=0.5, JPEG_Q=(60, 95), POS_CROP_PROB=0.7,
    WEIGHT_DECAY=1e-4, PATIENCE=5, VAL_PER_CLASS=150, TTA=True,
)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
CFG["LOC_BATCH"] = 12 if vram >= 14 else 6 if vram > 0 else 2
CFG["UNI_BATCH"] = 16 if vram >= 14 else 8 if vram > 0 else 4
CFG["NUM_WORKERS"] = 2 if IN_COLAB else min(4, os.cpu_count() or 2)    # Windows: set 0 if a worker error appears
LOC2, UNI2 = f'loc_v2_{CFG["ENCODER"]}', f'unified_v2_{CFG["ENCODER"]}'
DEPLOY_V1, DEPLOY_V2 = LOC_MODELS_DIR / "final_hybrid_deploy.pkl", LOC_MODELS_DIR / "final_hybrid_v2_deploy.pkl"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); USE_AMP = device.type == "cuda"
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
CLASS_NAMES = ["Real", "Full AI", "Partial AI"]
need = [DEPLOY_V1, LOC_MODELS_DIR / f'{CFG["V1_UNIFIED"]}_best.pth', LOC_MODELS_DIR / f'{CFG["V1_LOC"]}_best.pth',
        LOC_MODELS_DIR / f'{CFG["V1_LOC"]}_deploy.pth', PROJECT_DIR / "ood_test" / "partA" / "index.csv",
        PROJECT_DIR / "ood_test" / "partB" / "index.csv", LOC_FEATURES_DIR / "mask_audit.csv", PROJECT_DIR / "COCO_5000",
        PROJECT_DIR / "DiffusionDB_5000"]
missing = [str(p) for p in need if not p.exists()]
assert not missing, "Missing (run notebooks 13 and 14 first):\n  " + "\n  ".join(missing)
print("Device:", device, "| v2 runs:", LOC2, UNI2)

Device: cuda | v2 runs: loc_v2_resnet34 unified_v2_resnet34


In [2]:
# ============================================================
# 2. Main table (same as notebook 13): COCO, originals, DiffusionDB, Flux edits - source-photo split
# ============================================================
pairs = pd.read_csv(BASE_DIR / "pairs.csv"); pairs["pairs_row"] = np.arange(len(pairs))
audit = pd.read_csv(LOC_FEATURES_DIR / "mask_audit.csv").set_index("row")
a = audit.loc[pairs["pairs_row"].to_numpy()]
use_csv = (a["verdict"] == "ok").to_numpy()
use_alt = ((~use_csv) & (a["alt_verdict"] == "ok").to_numpy()) if "alt_verdict" in a else np.zeros(len(a), bool)
alt_paths = a["alt_mask_path"].astype(str).to_numpy() if "alt_mask_path" in a else pairs["mask_path"].to_numpy()
pairs["verified_mask"] = np.where(use_csv, pairs["mask_path"], np.where(use_alt, alt_paths, ""))
segm = pairs[pairs.mask_type == "segm"].reset_index(drop=True)
SRC_SPLIT = {}
for name in ("val", "test"):
    for s in segm["source_id"].to_numpy()[np.load(LOC_FEATURES_DIR / f"{name}_idx.npy")]:
        SRC_SPLIT[int(s)] = name
src_split = lambda s: SRC_SPLIT.get(int(s), "train")

rows = []
for r in pairs.itertuples():
    rows.append(dict(path=str(BASE_DIR / r.manipulated_path), label=2, kind="partial", mtype=r.mask_type,
                     mask=str(BASE_DIR / r.verified_mask) if r.verified_mask else "", split=src_split(r.source_id)))
for op, s in pairs.groupby("original_path")["source_id"].first().items():
    rows.append(dict(path=str(BASE_DIR / op), label=0, kind="original", mtype="", mask="", split=src_split(s)))
paths15 = np.load(FEATURES_DIR / "image_paths_15k.npy", allow_pickle=True); y15 = np.load(FEATURES_DIR / "y_labels_15k.npy")
split15 = np.empty(len(y15), dtype=object)
for name in ("train", "val", "test"): split15[np.load(FEATURES_DIR / f"{name}_idx.npy")] = name
def local(p):
    p = str(p).replace("\\", "/"); k = "AI_Image_Project/"
    return str(PROJECT_DIR / p.split(k, 1)[1]) if k in p else p
for i in np.where(y15 <= 1)[0]:
    p = local(paths15[i]); stem = Path(p).stem; sp = split15[i]
    if y15[i] == 0 and stem.isdigit() and int(stem) in SRC_SPLIT: sp = SRC_SPLIT[int(stem)]
    rows.append(dict(path=p, label=int(y15[i]), kind="coco" if y15[i] == 0 else "diffusiondb", mtype="", mask="", split=sp))
DF = pd.DataFrame(rows); DF = DF[DF["path"].map(lambda p: Path(p).exists())].reset_index(drop=True)
print(pd.crosstab(DF.kind, DF.split).to_string())

split        test  train  val
kind                         
coco          589   3817  594
diffusiondb   500   4000  500
original      113   1333  112
partial       429   4073  498


In [3]:
# ============================================================
# 3. New Full-AI data: COCO_AI rows NOT used in notebook 14 (real + SD 2.1 / SDXL / SD 3 only)
# ============================================================
from datasets import load_dataset
COLS = {"coco_image": "real_coco", "sd3_image": "sd3", "sd21_image": "sd2.1", "sdxl_image": "sdxl"}
def ahash(img, n=16):
    a_ = np.asarray(img.convert("L").resize((n, n), Image.Resampling.BILINEAR), np.float32); return (a_ > a_.mean()).flatten()
cocoai_dir = EXTRA / "cocoai"
if not (cocoai_dir / "index.csv").exists():
    known = []                                   # our own photos: never add one of them again as a 'new' real photo
    for f in tqdm(list((PROJECT_DIR / "COCO_5000").glob("*.jpg")) + list((BASE_DIR / "Original").glob("*.png")), desc="Hashing own photos"):
        try: known.append(ahash(Image.open(f)))
        except Exception: pass
    known = np.stack(known)
    stream = load_dataset("NasrinImp/COCO_AI", split="train", streaming=True).skip(CFG["COCOAI_SKIP_ROWS"])
    out, kept = [], 0
    for r in tqdm(stream, total=int(CFG["COCOAI_ROWS"] * 1.2), desc="Streaming COCO_AI"):
        if kept >= CFG["COCOAI_ROWS"]: break
        real = r.get("coco_image")
        if real is None or (np.abs(known ^ ahash(real)[None]).sum(1) <= 12).any(): continue
        split = "val" if kept % 7 == 0 else "train"
        for col, name in COLS.items():
            if name != "real_coco" and name not in CFG["TRAIN_GENERATORS"]: continue
            im = r.get(col)
            if im is None: continue
            d = cocoai_dir / name; d.mkdir(parents=True, exist_ok=True); p = d / f"{kept:04d}.png"; im.convert("RGB").save(p)
            out.append(dict(path=str(p), label=0 if name == "real_coco" else 1, kind="cocoai_real" if name == "real_coco" else "cocoai_fake",
                            mtype=name, mask="", split=split))
        kept += 1
    pd.DataFrame(out).to_csv(cocoai_dir / "index.csv", index=False)
COCOAI = pd.read_csv(cocoai_dir / "index.csv").fillna("")
print(pd.crosstab([COCOAI.kind, COCOAI.mtype], COCOAI.split).to_string())

split                  train  val
kind        mtype                
cocoai_fake sd2.1        514   86
            sd3          514   86
            sdxl         514   86
cocoai_real real_coco    514   86


In [4]:
# ============================================================
# 4. New Partial-AI data: Stable Diffusion 1.5 inpainting of TRAIN (and VAL) photos inside their verified masks
#    Resumable: already generated images are skipped.
# ============================================================
sd_dir = EXTRA / "sdinpaint"; sd_dir.mkdir(exist_ok=True)
ok_rows = pairs[(audit.loc[pairs.pairs_row, "verdict"].to_numpy() == "ok")].copy()      # verified mask, same-size original
ok_rows["split"] = ok_rows.source_id.map(src_split)
plan = pd.concat([ok_rows[ok_rows.split == "train"].sample(min(CFG["N_SD_TRAIN"], (ok_rows.split == "train").sum()), random_state=1),
                  ok_rows[ok_rows.split == "val"].sample(min(CFG["N_SD_VAL"], (ok_rows.split == "val").sum()), random_state=1)]).reset_index(drop=True)
todo = [k for k in range(len(plan)) if not (sd_dir / f"sd_{k:04d}.png").exists()]
print(f"SD edits planned: {len(plan)} | still to generate: {len(todo)}")
if todo:
    from diffusers import StableDiffusionInpaintPipeline
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32; pipe = None
    for repo in ["stable-diffusion-v1-5/stable-diffusion-inpainting", "runwayml/stable-diffusion-inpainting"]:
        for kw in ({"variant": "fp16"}, {}):
            try: pipe = StableDiffusionInpaintPipeline.from_pretrained(repo, torch_dtype=dtype, safety_checker=None, **kw); break
            except Exception as e: print(f"  could not load {repo} {kw}: {str(e)[:100]}")
        if pipe is not None: break
    assert pipe is not None, "Could not load the SD 1.5 inpainting model (notebook 14 downloaded it - check the internet connection)"
    pipe = pipe.to(device); pipe.enable_attention_slicing(); pipe.set_progress_bar_config(disable=True)
    rng = random.Random(CFG["SEED"])
    prompts = [rng.choice(CFG["SD_PROMPTS"]) for _ in range(len(plan))]
    for k in tqdm(todo, desc="SD inpainting"):
        r = plan.iloc[k]
        orig = Image.open(BASE_DIR / r.original_path).convert("RGB"); mask = Image.open(BASE_DIR / r.mask_path).convert("L")
        if mask.size != orig.size: mask = mask.resize(orig.size, Image.Resampling.NEAREST)
        W, H = orig.size; s = 512 / max(W, H); w8, h8 = max(64, int(W * s) // 8 * 8), max(64, int(H * s) // 8 * 8)
        g = torch.Generator(device=device).manual_seed(1000 + k)
        res = pipe(prompt=prompts[k], image=orig.resize((w8, h8), Image.Resampling.BICUBIC), mask_image=mask.resize((w8, h8), Image.Resampling.NEAREST),
                   num_inference_steps=25, guidance_scale=7.5, generator=g).images[0].resize((W, H), Image.Resampling.BICUBIC)
        m = np.asarray(mask) > 127; edited = np.asarray(orig).copy(); edited[m] = np.asarray(res)[m]
        mask.save(sd_dir / f"mask_{k:04d}.png"); Image.fromarray(edited).save(sd_dir / f"sd_{k:04d}.png")   # image last = complete
    del pipe; torch.cuda.empty_cache()
SD = pd.DataFrame([dict(path=str(sd_dir / f"sd_{k:04d}.png"), label=2, kind="sdinpaint", mtype="sd", mask=str(sd_dir / f"mask_{k:04d}.png"),
                        split=plan.iloc[k].split) for k in range(len(plan)) if (sd_dir / f"sd_{k:04d}.png").exists()])
print(SD.groupby("split").size().to_string())

DF2 = pd.concat([DF, COCOAI, SD], ignore_index=True)
DF2["has_mask"] = DF2["mask"] != ""
print("\nAll data (v2):"); print(pd.crosstab([DF2.kind, DF2.label], DF2.split, margins=True).to_string())

SD edits planned: 660 | still to generate: 0
split
train    600
val       60

All data (v2):


split              test  train   val    All
kind        label                          
coco        0       589   3817   594   5000
cocoai_fake 1         0   1542   258   1800
cocoai_real 0         0    514    86    600
diffusiondb 1       500   4000   500   5000
original    0       113   1333   112   1558
partial     2       429   4073   498   5000
sdinpaint   2         0    600    60    660
All                1631  15879  2108  19618


In [5]:
# ============================================================
# 5. Data loading module (same augmentations as notebook 13) + model / loss / evaluation helpers
# ============================================================
MODULE = r"""
import io, random
import numpy as np, torch
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
IGNORE = 255
def load_resized(path, max_side):
    im = Image.open(path).convert("RGB"); s = min(1.0, max_side / max(im.size))
    if s < 1: im = im.resize((max(1, round(im.size[0] * s)), max(1, round(im.size[1] * s))), Image.Resampling.BICUBIC)
    return im
def target_mask(label, mask_path, size):
    w, h = size
    if label == 0: return np.zeros((h, w), np.uint8)
    if label == 1: return np.ones((h, w), np.uint8)
    m = Image.open(mask_path).convert("L")
    if m.size != size: m = m.resize(size, Image.Resampling.NEAREST)
    return (np.asarray(m) > 127).astype(np.uint8)
def jpeg(img, q):
    b = io.BytesIO(); img.save(b, "JPEG", quality=int(q)); b.seek(0); return Image.open(b).convert("RGB")
class TrainSet(torch.utils.data.Dataset):
    def __init__(self, items, cfg, crop): self.items, self.c, self.crop = items, cfg, crop
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        for _ in range(10):
            try: return self._get(k)
            except Exception: k = random.randrange(len(self.items))
        raise RuntimeError("10 unreadable images in a row")
    def _get(self, k):
        path, label, mpath = self.items[k]; c = self.c; crop = self.crop
        img = load_resized(path, c["MAX_SIDE"])
        if random.random() < c["SCALE_PROB"]:
            s = random.uniform(*c["SCALE_RANGE"]); img = img.resize((max(1, round(img.size[0] * s)), max(1, round(img.size[1] * s))), Image.Resampling.BICUBIC)
        m = target_mask(label, mpath, img.size); a = np.asarray(img); H, W = m.shape
        if H < crop or W < crop:
            ph, pw = max(0, crop - H), max(0, crop - W)
            a = np.pad(a, ((0, ph), (0, pw), (0, 0))); m = np.pad(m, ((0, ph), (0, pw)), constant_values=IGNORE); H, W = m.shape
        if label == 2 and random.random() < c["POS_CROP_PROB"] and (m == 1).any():
            ys, xs = np.nonzero(m == 1); j = random.randrange(len(ys)); y, x = ys[j], xs[j]
            y0 = int(np.clip(y - random.randrange(crop), 0, H - crop)); x0 = int(np.clip(x - random.randrange(crop), 0, W - crop))
        else:
            y0, x0 = random.randint(0, H - crop), random.randint(0, W - crop)
        a = a[y0:y0 + crop, x0:x0 + crop]; m = m[y0:y0 + crop, x0:x0 + crop]
        if random.random() < 0.5: a, m = a[:, ::-1], m[:, ::-1]
        if random.random() < c["JPEG_PROB"]: a = np.asarray(jpeg(Image.fromarray(np.ascontiguousarray(a)), random.randint(*c["JPEG_Q"])))
        return (torch.from_numpy(np.ascontiguousarray(a).transpose(2, 0, 1).copy()), torch.from_numpy(np.ascontiguousarray(m).copy()), torch.tensor(label))
def worker_init(i):
    s = torch.initial_seed() % 2**32; random.seed(s + i); np.random.seed((s + i) % 2**32)
"""
Path("unified_ds_v2.py").write_text(MODULE)
if str(Path.cwd()) not in sys.path: sys.path.insert(0, str(Path.cwd()))
import unified_ds_v2 as uds; importlib.reload(uds)

def make_loader(df, crop, samples, shares, batch):
    items = list(zip(df.path, df.label, df["mask"]))
    w = (df.kind.map(shares) / df.kind.map(df.kind.value_counts())).fillna(0).to_numpy()
    sampler = torch.utils.data.WeightedRandomSampler(torch.tensor(w, dtype=torch.double), samples, replacement=True)
    return torch.utils.data.DataLoader(uds.TrainSet(items, CFG, crop), batch_size=batch, sampler=sampler, num_workers=CFG["NUM_WORKERS"],
                                       pin_memory=USE_AMP, drop_last=True, persistent_workers=CFG["NUM_WORKERS"] > 0, worker_init_fn=uds.worker_init)

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1) * 255
THRS = np.round(np.r_[0.01, 0.02, 0.03, 0.04, np.arange(0.05, 0.951, 0.05)], 2)
def make_model(arch=None, enc=None):
    return smp.create_model(arch=arch or CFG["ARCH"], encoder_name=enc or CFG["ENCODER"], encoder_weights=None, in_channels=3, classes=1).to(device).to(memory_format=torch.channels_last)
def load_weights(model, path):
    model.load_state_dict(torch.load(path, map_location=device, weights_only=False)["model"]); return model.eval()
def norm(x): return ((x.to(device, non_blocking=True).float() - MEAN) / STD).contiguous(memory_format=torch.channels_last)
def unified_loss(logits, target, labels):
    logits = logits.float(); valid = (target != 255).float(); t = (target == 1).float()
    bce = (F.binary_cross_entropy_with_logits(logits, t, reduction="none") * valid).sum() / valid.sum().clamp(min=1)
    part = (labels == 2).view(-1, 1, 1, 1).float() * valid
    if part.sum() == 0: return bce
    p = torch.sigmoid(logits); dice = (2 * (p * t * part).sum() + 1) / ((p * part).sum() + (t * part).sum() + 1)
    return bce + (1 - dice)
@torch.no_grad()
def predict_prob(model, img_arr, tta=False):
    H, W = img_arr.shape[:2]
    x = norm(torch.from_numpy(np.array(img_arr)).permute(2, 0, 1)[None]); x = F.pad(x, (0, (-W) % 32, 0, (-H) % 32), mode="reflect")
    with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
        p = torch.sigmoid(model(x).float())
        if tta: p = 0.5 * (p + torch.sigmoid(model(x.flip(-1)).float()).flip(-1))
    return p[0, 0, :H, :W].float()
def image_stats(p):
    f = p.flatten(); k = max(1, int(0.01 * f.numel()))
    q = torch.quantile(f[:: max(1, f.numel() // 200000)], torch.tensor([0.99, 0.95, 0.5], device=f.device))
    return np.array([(f > t).float().mean().item() for t in (0.1, 0.3, 0.5, 0.7, 0.9)] + [f.mean().item(), f.std().item(), f.max().item()]
                    + q.tolist() + [f.topk(k).values.mean().item()], np.float32)
@torch.no_grad()
def run_eval(model, df, tta=False, desc="eval"):
    model.eval(); torch.backends.cudnn.benchmark = False
    thr = torch.tensor(THRS, device=device, dtype=torch.float32).view(-1, 1, 1)
    keep, S, inter, pred, gt, npx, fracs = [], [], [], [], [], [], []
    for j, r in enumerate(tqdm(df.itertuples(), total=len(df), desc=desc, leave=False)):
        try: img = uds.load_resized(r.path, CFG["MAX_SIDE"])
        except Exception: continue
        p = predict_prob(model, np.asarray(img), tta); keep.append(j); S.append(image_stats(p)); npx.append(p.numel())
        fracs.append((p.unsqueeze(0) >= thr).float().mean((1, 2)).cpu().numpy())
        if r.label == 2 and r.mask:
            g = torch.from_numpy(uds.target_mask(2, r.mask, img.size)).to(device).bool(); pr = p.unsqueeze(0) >= thr
            inter.append((pr & g).sum((1, 2)).cpu().numpy()); pred.append(pr.sum((1, 2)).cpu().numpy()); gt.append(g.sum().item())
        else:
            inter.append(None); pred.append(None); gt.append(None)
    torch.backends.cudnn.benchmark = True
    sub = df.iloc[keep].reset_index(drop=True)
    return dict(df=sub, stats=np.stack(S), label=sub.label.to_numpy(), inter=inter, pred=pred, gt=gt, npx=np.array(npx, float), fracs=np.stack(fracs))
def pixel_f1(ev, t, kind=None, mtype=None):
    k = [i for i, g in enumerate(ev["gt"]) if g is not None and (kind is None or ev["df"].kind[i] == kind) and (mtype is None or ev["df"].mtype[i] == mtype)]
    if not k: return np.nan, np.nan, 0
    I = np.array([ev["inter"][i][t] for i in k]); P = np.array([ev["pred"][i][t] for i in k]); G = np.array([ev["gt"][i] for i in k])
    return float((2 * I / (P + G + 1e-9)).mean()), float(2 * I.sum() / (P.sum() + G.sum() + 1e-9)), len(k)
def make_clf(): return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
def cv_class_f1(ev):
    n_min = np.bincount(ev["label"], minlength=3).min()
    pred = cross_val_predict(make_clf(), ev["stats"], ev["label"], cv=StratifiedKFold(min(5, max(2, n_min)), shuffle=True, random_state=0))
    return f1_score(ev["label"], pred, average="macro")

def train_phase(name, model, loader, epochs, lr, val_fn):
    # generic fine-tuning loop with best/last checkpoints, auto-resume and early stopping
    best_p, last_p = LOC_MODELS_DIR / f"{name}_best.pth", LOC_MODELS_DIR / f"{name}_last.pth"
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG["WEIGHT_DECAY"])
    steps = len(loader); total = epochs * steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / steps) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total)))))
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    start, best, bad, hist = 0, -1.0, 0, []
    if last_p.exists():
        ck = torch.load(last_p, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        start, best, bad, hist = ck["epoch"], ck["best"], ck["bad"], ck["hist"]; print(f"{name}: resumed after epoch {start}")
    else:
        s0 = val_fn(model); best = s0["score"]; torch.save({"model": model.state_dict(), "epoch": 0}, best_p)
        print(f"{name}: starting point (v1 weights) validation score {best:.4f} " + str({k: round(v, 4) for k, v in s0.items()}))
    torch.backends.cudnn.benchmark = True
    for ep in range(start, epochs):
        if bad >= CFG["PATIENCE"]: print("Early stopping."); break
        model.train(); t0 = time.time(); tot = 0.0
        for x, y, lab in tqdm(loader, desc=f"{name} epoch {ep+1}/{epochs}", leave=False):
            y = y.to(device, non_blocking=True).unsqueeze(1); lab = lab.to(device); opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP): logits = model(norm(x))
            loss = unified_loss(logits, y, lab); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
        s = val_fn(model); improved = s["score"] > best
        hist.append(dict(epoch=ep + 1, loss=tot / steps, **s, minutes=(time.time() - t0) / 60))
        if improved: best, bad = s["score"], 0; torch.save({"model": model.state_dict(), "epoch": ep + 1}, best_p)
        else: bad += 1
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": ep + 1, "best": best, "bad": bad, "hist": hist}, last_p)
        print(f"{name} ep {ep+1:02d} | loss {tot/steps:.4f} | " + " | ".join(f"{k} {v:.4f}" for k, v in s.items()) + f" | {(time.time()-t0)/60:.1f} min" + ("  *best*" if improved else ""))
    pd.DataFrame(hist).to_csv(OUT_DIR / f"{name}_history.csv", index=False)
    return load_weights(model, best_p)
print("Helpers ready")

Helpers ready


In [6]:
# ============================================================
# 6. Your images: prepare (upright + resize like the app), group near-duplicates, split
# ============================================================
from PIL import ImageOps
try:
    ensure("pillow_heif", "pillow-heif"); from pillow_heif import register_heif_opener; register_heif_opener()
except Exception as e:
    print("HEIC support not available (only matters for iPhone .heic files):", str(e)[:80])
OWN = PROJECT_DIR / "own_data"; PREP = OWN / "_prepared"
OWN_SRC = {"phone_real": (OWN / "phone_real", 0), "chatgpt": (OWN / "chatgpt", 1)}
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".heic", ".heif", ".bmp", ".tif", ".tiff"}
rows = []
for kind, (d, lab) in OWN_SRC.items():
    d.mkdir(parents=True, exist_ok=True); (PREP / kind).mkdir(parents=True, exist_ok=True)
    files = sorted(f for f in d.rglob("*") if f.is_file() and f.suffix.lower() in EXTS)
    for f in tqdm(files, desc=f"preparing {kind}"):
        out = PREP / kind / (f.relative_to(d).as_posix().replace("/", "__") + ".png")
        if not out.exists():
            try:
                im = ImageOps.exif_transpose(Image.open(f)).convert("RGB")          # same steps as app.prepare()
                s = min(1.0, CFG["MAX_SIDE"] / max(im.size))
                if s < 1: im = im.resize((round(im.size[0] * s), round(im.size[1] * s)), Image.Resampling.BICUBIC)
                im.save(out)
            except Exception as e:
                print("  skipped", f.name, "-", str(e)[:80]); continue
        rows.append(dict(path=str(out), label=lab, kind=kind, mtype=kind, mask="", src=f.name))
OWN_DF = pd.DataFrame(rows, columns=["path", "label", "kind", "mtype", "mask", "src"])
counts = OWN_DF.kind.value_counts().to_dict(); print("Your images:", counts)
assert counts.get("phone_real", 0) >= 20 and counts.get("chatgpt", 0) >= 20, \
    "Need at least 20 images in each of own_data/phone_real and own_data/chatgpt (better 150+ and 80+)"
if counts["phone_real"] < 60 or counts["chatgpt"] < 40: print("WARNING: few images - results will be noisy. Aim for 150+ phone photos and 80+ ChatGPT images.")

def dhash(p):
    a = np.asarray(Image.open(p).convert("L").resize((9, 8), Image.Resampling.BILINEAR), np.int16); return (a[:, 1:] > a[:, :-1]).flatten()
Hs = np.stack([dhash(p) for p in OWN_DF.path]); parent = list(range(len(Hs)))
def root(i):
    while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
    return i
D = (Hs[:, None, :] != Hs[None, :, :]).sum(2)
for i, j in zip(*np.where(np.triu(D <= 8, 1))): parent[root(i)] = root(j)
OWN_DF["group"] = [root(i) for i in range(len(Hs))]
u = {g: random.Random(f"{CFG['SEED']}-{g}-{OWN_DF.src[OWN_DF.group == g].min()}").random() for g in OWN_DF.group.unique()}
OWN_DF["split"] = "train"                                    # per class, whole groups in random order: first 25% -> test, next 20% -> val
gk = OWN_DF.groupby("group").kind.agg(lambda s: s.mode()[0])
for kind in OWN_SRC:
    n_k, done = int((OWN_DF.kind == kind).sum()), 0
    for g in sorted(gk[gk == kind].index, key=lambda g: u[g]):
        OWN_DF.loc[OWN_DF.group == g, "split"] = "test" if done < 0.25 * n_k else "val" if done < 0.45 * n_k else "train"
        done += int((OWN_DF.group == g).sum())
print(f"Near-duplicate groups: {OWN_DF.group.nunique()} for {len(OWN_DF)} images")
print(pd.crosstab(OWN_DF.kind, OWN_DF.split).to_string())
ct = pd.crosstab(OWN_DF.kind, OWN_DF.split).reindex(columns=["train", "val", "test"], fill_value=0)
assert (ct.values >= 3).all(), "Too few images in some split - add more (and more varied) images and run again"
OWN_DF.to_csv(OWN / "own_split.csv", index=False)

preparing phone_real:   0%|          | 0/63 [00:00<?, ?it/s]

preparing chatgpt:   0%|          | 0/40 [00:00<?, ?it/s]

Your images: {'phone_real': 63, 'chatgpt': 40}


Near-duplicate groups: 81 for 103 images
split       test  train  val
kind                        
chatgpt        7     27    6
phone_real    20     29   14


In [7]:
# ============================================================
# 7. Fine-tune the unified model: v2 weights + all old sources + your images -> v3
#    (restarts automatically when the set of your images changes)
# ============================================================
UNI3 = f'unified_v3_{CFG["ENCODER"]}'
DEPLOY_C, DEPLOY_V3 = LOC_MODELS_DIR / "final_hybrid_combo_deploy.pkl", LOC_MODELS_DIR / "final_hybrid_v3_deploy.pkl"
assert DEPLOY_C.exists() and (OUT_DIR / "v1_v2_combo_per_image.csv").exists(), "Run notebook 16 first"
d1, dc = pickle.load(open(DEPLOY_V1, "rb")), pickle.load(open(DEPLOY_C, "rb"))
V3_EPOCHS, V3_LR = int(os.environ.get("V3_EPOCHS", 8)), 5e-5
sig_file = LOC_MODELS_DIR / f"{UNI3}_data.json"; sig = sorted(OWN_DF.path.map(lambda p: Path(p).name) + "|" + OWN_DF.split)
if sig_file.exists() and json.loads(sig_file.read_text()) != sig:
    print("Your images changed since the last run -> training v3 again from the v2 weights")
    for suf in ("_best.pth", "_last.pth"): (LOC_MODELS_DIR / f"{UNI3}{suf}").unlink(missing_ok=True)
sig_file.write_text(json.dumps(sig))

DF3 = pd.concat([DF2.drop(columns=["has_mask"], errors="ignore"), OWN_DF[["path", "label", "kind", "mtype", "mask", "split"]]], ignore_index=True)
DF3["has_mask"] = DF3["mask"] != ""
uni_train = DF3[(DF3.split == "train") & ((DF3.label != 2) | DF3.has_mask)].reset_index(drop=True)
uni_val = DF3[DF3.split == "val"].reset_index(drop=True)
quick = pd.concat([d.sample(min(len(d), CFG["VAL_PER_CLASS"] // 2), random_state=0) for _, d in uni_val.groupby("kind")]).reset_index(drop=True)
SHARE3 = {"coco": 1/12, "original": 1/12, "cocoai_real": 1/12, "phone_real": 1/6,
          "diffusiondb": 1/8, "cocoai_fake": 1/8, "chatgpt": 1/6, "partial": 1/8, "sdinpaint": 1/8}
print("Training images:", uni_train.kind.value_counts().to_dict())
OWN_KINDS = ["phone_real", "chatgpt"]

def uni3_val_fn(model):
    ev = run_eval(model, quick, desc="unified val"); y = ev["label"]; kinds = ev["df"].kind.to_numpy()
    n_min = np.bincount(y, minlength=3).min()
    pred = cross_val_predict(make_clf(), ev["stats"], y, cv=StratifiedKFold(min(5, max(2, n_min)), shuffle=True, random_state=0))
    c = f1_score(y, pred, average="macro"); p = np.nanmax([pixel_f1(ev, t)[0] for t in range(len(THRS))])
    oa = float(np.mean([(pred[kinds == k] == y[kinds == k]).mean() for k in OWN_KINDS]))
    return {"score": (c + p + oa) / 3, "class_macroF1": c, "pixelF1": p, "own_acc": oa}

uni3 = load_weights(make_model(), LOC_MODELS_DIR / f"{dc['unified_runs'][1]}_best.pth")       # start from v2 unified
uni3 = train_phase(UNI3, uni3, make_loader(uni_train, CFG["UNI_CROP"], CFG["UNI_SAMPLES"], SHARE3, CFG["UNI_BATCH"]), V3_EPOCHS, V3_LR, uni3_val_fn)

Training images: {'diffusiondb': 4000, 'coco': 3817, 'partial': 2139, 'cocoai_fake': 1542, 'original': 1333, 'sdinpaint': 600, 'cocoai_real': 514, 'phone_real': 29, 'chatgpt': 27}


unified val:   0%|          | 0/530 [00:00<?, ?it/s]

unified_v3_resnet34: starting point (v1 weights) validation score 0.5144 {'score': np.float64(0.5144), 'class_macroF1': 0.8198, 'pixelF1': np.float64(0.509), 'own_acc': 0.2143}


unified_v3_resnet34 epoch 1/8:   0%|          | 0/375 [00:10<?, ?it/s]

unified val:   0%|          | 0/530 [00:00<?, ?it/s]

unified_v3_resnet34 ep 01 | loss 1.0762 | score 0.5315 | class_macroF1 0.7824 | pixelF1 0.5145 | own_acc 0.2976 | 2.8 min  *best*


... (11 outputs left out)

unified_v3_resnet34 ep 05 | loss 0.6930 | score 0.4883 | class_macroF1 0.7750 | pixelF1 0.4638 | own_acc 0.2262 | 2.3 min


unified_v3_resnet34 epoch 6/8:   0%|          | 0/375 [00:00<?, ?it/s]

unified val:   0%|          | 0/530 [00:00<?, ?it/s]

unified_v3_resnet34 ep 06 | loss 0.6433 | score 0.5222 | class_macroF1 0.7730 | pixelF1 0.5198 | own_acc 0.2738 | 2.3 min
Early stopping.


In [8]:
# ============================================================
# 8. Refit the decision on VALIDATION (v1 unified, v1 edit finder, v3 unified, v2 edit finder) - same rules as notebook 16
# ============================================================
def load_loc(run):
    d = torch.load(LOC_MODELS_DIR / f"{run}_deploy.pth", map_location=device, weights_only=False)
    m = make_model(d["cfg"]["ARCH"], d["cfg"]["ENCODER"]); m.load_state_dict(d["model"]); return m.eval()
NETS = {"uni1": load_weights(make_model(), LOC_MODELS_DIR / f"{dc['unified_runs'][0]}_best.pth"), "loc1": load_loc(dc["loc_runs"][0]),
        "uni2": load_weights(make_model(), LOC_MODELS_DIR / f"{dc['unified_runs'][1]}_best.pth"), "loc2": load_loc(dc["loc_runs"][1]), "uni3": uni3.eval()}
ORDER_C, ORDER_3 = ["uni1", "loc1", "uni2", "loc2"], ["uni1", "loc1", "uni3", "loc2"]

@torch.no_grad()
def stats_of(df, names, desc):
    keep, S = [], {n: [] for n in names}
    for j, r in enumerate(tqdm(df.itertuples(), total=len(df), desc=desc)):
        try: img = uds.load_resized(r.path, CFG["MAX_SIDE"])
        except Exception: continue
        arr = np.asarray(img); keep.append(j)
        for n in names: S[n].append(image_stats(predict_prob(NETS[n], arr, True)))
    return df.iloc[keep].reset_index(drop=True), {n: np.stack(v) for n, v in S.items()}
def feats(S, order): return np.hstack([S[n] for n in order])

vdf, VS = stats_of(DF3[DF3.split == "val"].reset_index(drop=True), ORDER_3, "validation (v3 system)")
Xv, yv, kv = feats(VS, ORDER_3), vdf.label.to_numpy(), vdf.kind.to_numpy()
cnt = pd.Series(kv).value_counts(); W_SRC = (len(kv) / (len(cnt) * pd.Series(kv).map(cnt))).to_numpy()
cv = StratifiedKFold(min(5, max(2, np.bincount(yv, minlength=3).min())), shuffle=True, random_state=0)
def fit_clf(X, y, w): return make_clf().fit(X, y, logisticregression__sample_weight=w)
FEATSETS = {"v1 networks": list(range(0, 24)), "v3 unified + v2 edit finder": list(range(24, 48)), "all four": list(range(0, 48))}
fs_rows = []
for name, idx in FEATSETS.items():
    for wname, w in (("plain", None), ("source-balanced", W_SRC)):
        pred = np.empty_like(yv)
        for tr, te in cv.split(Xv, yv): pred[te] = fit_clf(Xv[tr][:, idx], yv[tr], None if w is None else w[tr]).predict(Xv[te][:, idx])
        per_src = pd.Series(pred == yv).groupby(kv).mean(); mf1 = f1_score(yv, pred, average="macro")
        fs_rows.append(dict(features=name, weights=wname, cv_macroF1=mf1, mean_source_acc=per_src.mean(), score=0.5 * mf1 + 0.5 * per_src.mean(),
                            **{f"acc {k}": v for k, v in per_src.items()}))
fs_tab = pd.DataFrame(fs_rows).set_index(["features", "weights"]); print(fs_tab.round(3).to_string())
FEAT_NAME, W_NAME = fs_tab.score.idxmax(); FEAT_IDX = FEATSETS[FEAT_NAME]
CLF3 = fit_clf(Xv[:, FEAT_IDX], yv, W_SRC if W_NAME == "source-balanced" else None)
print(f"\nChosen on validation -> {FEAT_NAME} ({W_NAME} weights); mask and area exactly as v1+v2 ({dc['blend']})")

validation (v3 system):   0%|          | 0/2128 [00:00<?, ?it/s]

                                             cv_macroF1  mean_source_acc  score  acc chatgpt  acc coco  acc cocoai_fake  acc cocoai_real  acc diffusiondb  acc original  acc partial  acc phone_real  acc sdinpaint
features                    weights                                                                                                                                                                                
v1 networks                 plain                 0.813            0.618  0.716        0.167     0.875            0.752            0.849            1.000         0.607        0.735           0.143          0.433
                            source-balanced       0.760            0.623  0.691        0.167     0.887            0.550            0.872            1.000         0.732        0.604           0.643          0.150
v3 unified + v2 edit finder plain                 0.850            0.695  0.772        0.167     0.886            0.953            0.826            0.99

In [9]:
# ============================================================
# 9. Test: v1+v2 vs v3 - main test set, notebook 14 sets, and YOUR held-out test images
# ============================================================
test = DF[DF.split == "test"].reset_index(drop=True).assign(group=lambda d: d.kind)
pa = pd.read_csv(PROJECT_DIR / "ood_test" / "partA" / "index.csv")
pa = pd.DataFrame(dict(path=pa.path, label=np.where(pa.expected == 0, 0, 1), mask="",
                       group=[("unseen gen: " if s in ("sd3.5", "dalle3", "midjourney6") else ("real: " if s == "real_coco" else "seen gen: ")) + s for s in pa.source]))
pb = pd.read_csv(PROJECT_DIR / "ood_test" / "partB" / "index.csv")
pbd = pd.concat([pd.DataFrame(dict(path=pb.sd_edit, label=2, mask=pb["mask"], group="SD inpaint edit")),
                 pd.DataFrame(dict(path=pb.flux_edit, label=2, mask=pb["mask"], group="Flux edit")),
                 pd.DataFrame(dict(path=pb.original, label=0, mask="", group="untouched original"))], ignore_index=True)
own_test = OWN_DF[OWN_DF.split == "test"].reset_index(drop=True).assign(group=lambda d: d.kind.map({"phone_real": "YOUR phone photos (Real)", "chatgpt": "YOUR ChatGPT images (AI)"}))

rows = []
for df, tag in ((test, "main test"), (pa, "unseen generators"), (pbd, "SD vs Flux edits")):
    sub, S = stats_of(df, ORDER_3, tag)
    rows.append(pd.DataFrame(dict(version="v3", group=sub.group, label=sub.label, pred=CLF3.predict(feats(S, ORDER_3)[:, FEAT_IDX]))))
sub, S = stats_of(own_test, sorted(set(ORDER_C + ORDER_3)), "your test images")
rows.append(pd.DataFrame(dict(version="v3", group=sub.group, label=sub.label, pred=CLF3.predict(feats(S, ORDER_3)[:, FEAT_IDX]))))
rows.append(pd.DataFrame(dict(version="v1+v2", group=sub.group, label=sub.label, pred=dc["classifier"].predict(feats(S, ORDER_C)[:, dc["feat_idx"]]))))
old = pd.read_csv(OUT_DIR / "v1_v2_combo_per_image.csv"); old = old[old.version == "v1+v2"][["version", "group", "label", "pred"]]
res = pd.concat([old] + rows, ignore_index=True); res.to_csv(OUT_DIR / "v3_per_image.csv", index=False)

VERS = ["v1+v2", "v3"]
tab = (res.assign(correct=lambda d: (d.pred == d.label).astype(float), flagged_ai=lambda d: (d.pred > 0).astype(float))
          .groupby(["group", "version"]).agg(images=("correct", "size"), correct=("correct", "mean"), flagged_as_AI=("flagged_ai", "mean")).unstack("version"))
tab = tab.reindex(columns=pd.MultiIndex.from_product([["images", "correct", "flagged_as_AI"], VERS]))
print("=" * 100); print("v1+v2 (current final) vs v3 (with your images)"); print("=" * 100); print(tab.round(3).to_string())
tab.to_csv(OUT_DIR / "v3_summary.csv")
MAIN = ["coco", "original", "diffusiondb", "partial"]; OWNG = list(own_test.group.unique())
summ = []
for v in VERS:
    r_ = res[res.version == v]; m = r_[r_.group.isin(MAIN)]; o = r_[~r_.group.isin(MAIN + OWNG)]
    summ.append(dict(version=v, main_test_accuracy=(m.pred == m.label).mean(), main_test_macroF1=f1_score(m.label, m.pred, average="macro"),
                     new_sets_accuracy=(o.pred == o.label).mean(),
                     your_phone_photos=(lambda q: (q.pred == q.label).mean())(r_[r_.group == "YOUR phone photos (Real)"]),
                     your_chatgpt_images=(lambda q: (q.pred == q.label).mean())(r_[r_.group == "YOUR ChatGPT images (AI)"])))
summ = pd.DataFrame(summ).set_index("version"); summ.to_csv(OUT_DIR / "v3_overall.csv")
print("\nOverall:"); print(summ.round(3).to_string())
print(f"\nYour test images (never used for training or choices): {own_test.kind.value_counts().to_dict()}")

main test:   0%|          | 0/1631 [00:00<?, ?it/s]

unseen generators:   0%|          | 0/350 [00:00<?, ?it/s]

SD vs Flux edits:   0%|          | 0/180 [00:00<?, ?it/s]

your test images:   0%|          | 0/27 [00:00<?, ?it/s]

v1+v2 (current final) vs v3 (with your images)
                         images      correct        flagged_as_AI       
                          v1+v2   v3   v1+v2     v3         v1+v2     v3
group                                                                   
Flux edit                    60   60   0.600  0.600         0.617  0.633
SD inpaint edit              60   60   0.633  0.617         0.650  0.650
YOUR ChatGPT images (AI)      7    7   0.000  0.000         0.286  0.143
YOUR phone photos (Real)     20   20   0.450  0.400         0.550  0.600
coco                        589  589   0.925  0.925         0.075  0.075
diffusiondb                 500  500   0.996  0.996         0.996  0.996
original                    113  113   0.708  0.699         0.292  0.301
partial                     429  429   0.662  0.634         0.674  0.678
real: real_coco              50   50   0.880  0.940         0.120  0.060
seen gen: sd2.1              50   50   0.960  0.960         0.980  0.980
seen

In [10]:
# ============================================================
# 10. Save v3 for the app (python app.py --v3)
# ============================================================
pickle.dump({"classifier": CLF3, "combo": True, "feat_idx": FEAT_IDX, "feature_set": FEAT_NAME, "weights": W_NAME,
             "unified_runs": [dc["unified_runs"][0], UNI3], "loc_runs": dc["loc_runs"], "loc_weight": dc["loc_weight"], "blend": dc["blend"],
             "mask_t": dc["mask_t"], "area_t": dc["area_t"], "cfg": dc["cfg"], "unified_run": UNI3, "loc07_run": dc["loc07_run"]}, open(DEPLOY_V3, "wb"))
print("Saved", DEPLOY_V3.name, "-> start the app with:  python app.py --v3")

Saved final_hybrid_v3_deploy.pkl -> start the app with:  python app.py --v3
